<a href="https://colab.research.google.com/github/CrampsP/CrampsP.github.io/blob/main/Another_copy_of_True_starter_housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-learn==1.5.2
!pip install lets-plot

import pandas as pd
import numpy as np
from lets_plot import *
from sklearn.preprocessing import MinMaxScaler


In [ ]:
url = 'https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing.csv'
house = pd.read_csv(url)

housing = pd.read_csv(url)
housing

In [ ]:
#we could change all of the 0's in year renovated to the year it was built and see if that helps any
#         nevermind it didn't work :(

In [ ]:
def clean_housing(housing):
  housing['datetime'] = housing['date'].str.split('T').str[0]

  housing = housing.assign(
      year = housing['datetime'].str[0:4].astype(int),
      month = housing['datetime'].str[4:6].astype(int),

  )

  housing = housing.drop(columns=['date'])
  housing = housing.drop(columns=['datetime'])
  housing = housing.drop(columns=['id'])


  clean_house = housing
  clean_house = clean_house[clean_house['bedrooms'] <= 15]
  clean_house = clean_house[(clean_house['bathrooms'] <= 5.5) & (clean_house['bathrooms'] >= .5)]
  clean_house = clean_house[clean_house['sqft_living'] <= 8670]
  clean_house = clean_house[clean_house['sqft_lot'] <= 500000]
  clean_house['yr_renovated'] = clean_house.apply(
    lambda row: row['year'] if row['yr_renovated'] == 0 else row['yr_renovated'],
    axis=1
)

  return clean_house


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import MinMaxScaler


clean_house = clean_housing(housing)


y = clean_house['price']

X = clean_house.drop(columns=['price'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBRegressor()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

results_draft = pd.DataFrame(predictions, columns = ['predictions'])
results_draft.to_csv("bet-ren-predictions.csv",index=False)
draft = pd.read_csv("bet-ren-predictions.csv")
draft

In [ ]:
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error, r2_score

RMSE = root_mean_squared_error(y_test, predictions)
print(f'RMSE: {RMSE:.2f}')

MAE = mean_absolute_error(y_test, predictions)
print(f'MAE: {MAE:.2f}')

R2 = r2_score(y_test, predictions)
print(f'R2: {R2:.4f}')


# RMSE: 113937.57
# MAE: 67262.06
# R2: 0.8997


In [ ]:
#after all of my code above I imported the mini hold out set and call it the same thing here so I don't have to rename my copy paste code
url_mini = 'https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing_holdout_test_mini.csv'
housing_mini = pd.read_csv(url_mini)

clean_house_mini = clean_housing(housing_mini)
# I then take my cleaned data and send it through this code.
#Since I've already made the model above I don't have to redefine what the here.
X = clean_house_mini

predictions = model.predict(X)
#I export it
results_mini = pd.DataFrame(predictions, columns = ['predictions'])
results_mini.to_csv("bet-ren-predictions.csv",index=False)
bet_ren = pd.read_csv("bet-ren-predictions.csv")
bet_ren

In [ ]:
#after all of my code above I imported the mini hold out set and call it the same thing here so I don't have to rename my copy paste code
url_data = 'https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing_holdout_test.csv'
housing_data = pd.read_csv(url_data)

clean_house_data = clean_housing(housing_data)
# I then take my cleaned data and send it through this code.
#Since I've already made the model above I don't have to redefine what the here.
X = clean_house_mini

predictions = model.predict(X)
#I export it
results_data = pd.DataFrame(predictions, columns = ['predictions'])
results_data.to_csv("final-predictions.csv",index=False)
final = pd.read_csv("final-predictions.csv")
final

In [ ]:
# @title predictions

from matplotlib import pyplot as plt
final['predictions'].plot(kind='line', figsize=(8, 4), title='predictions')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
# prompt: convert this code into something that the "final" df can use but dont include price
#  sns.set(rc={'figure.figsize':(11.7,8.27)})
# for team_results in results_dict.items():
#   testfinal = team_results[1]['dataset']
#   # print(team_results['dataset'])
#   # print(f"R-Squared Value: {r2}")
#   print(f"-------------------------------- {team_results[0].upper()} RESULTS ---------------------------------\n")
#   print(f" Within 5%: {team_results[1]['Percent of houses within 5 percent']}%\n",
#   f"Within 10%: {team_results[1]['Percent of houses within 10 percent']}%\n",
#   f"Within 20%: {team_results[1]['Percent of houses within 20 percent']}%\n",
#   f"R^2: {team_results[1]['R2']}%\n",
#   f"RMSE: {team_results[1]['RMSE']}\n",
#   f"Mean Absolute Error: {team_results[1]['Mean Abs Error']}\n",
#   f"Median Aboslute Error: {team_results[1]['Median Abs Error']}")
#   color_dict = dict({'below 20%':'tab:blue',
#                     'above 20%': 'tab:orange'})
#   # print(testfinal['abspercentmiss'].describe(percentiles=[.1,.2,.3,.4,.5,.6,.7,.8,.9,.95]))
#   xlims=(0,4e6)
#   ylims=(0,4e6)
#   ax = sns.scatterplot(data=testfinal,x='price',y='predictions',hue="percent_bucket",palette=color_dict)
#   # ax.set(xscale="log", yscale="log", xlim=xlims, ylim=ylims)
#   ax.plot(xlims,xlims, color='r')
#   # plt.legend(labels=['perfect',"below 5",'above 5','10-20%','above 20'])
#   plt.show()
#   print(f"-"*77)
#   print("\n"*3)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(rc={'figure.figsize':(11.7,8.27)})

# Assuming 'final' DataFrame exists and has 'predictions' column
# You might need to add actual prices if you want to calculate performance metrics
# and create percent_bucket for the final dataset.

# Example structure if you have a 'price' column in the final dataset (for comparison/plotting)
# final['abs_error'] = abs(final['price'] - final['predictions'])
# final['abs_percent_error'] = (final['abs_error'] / final['price']) * 100
# final['percent_bucket'] = pd.cut(final['abs_percent_error'],
#                                  bins=[-float('inf'), 5, 10, 20, float('inf')],
#                                  labels=['below 5%', '5-10%', '10-20%', 'above 20%'])

# If you want to plot the predictions themselves without comparing to a known price
plt.figure(figsize=(11.7, 8.27))
ax = sns.lineplot(data=final, y='predictions', x=final.index) # Using index as x-axis
ax.set_title('Predictions')
ax.set_ylabel('Predicted Price')
ax.set_xlabel('Sample Index')
plt.show()

# If you want to see the distribution of predictions
plt.figure(figsize=(11.7, 8.27))
sns.histplot(final['predictions'], kde=True)
plt.title('Distribution of Predictions')
plt.xlabel('Predicted Price')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# prompt: write feature importance code for the clean_house dataset

import pandas as pd
import matplotlib.pyplot as plt

feature_importances = model.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({'feature': feature_names, 'importance': feature_importances})
importance_df = importance_df.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.bar(importance_df['feature'], importance_df['importance'])
plt.xticks(rotation=90)
plt.xlabel('Features')
plt.ylabel('Importance')
plt.title('Feature Importance from XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
# prompt: plot long against lat in a scatter plot but then differentiate price using the house df, limit price to under 1,000,000

ggplot(data=house[house['price'] < 250000], mapping=aes(x='long', y='lat', color='price')) + geom_point() + scale_color_gradient()